### Imports

In [2]:
import os
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from skmultilearn.model_selection import iterative_train_test_split 
from sklearn.model_selection import GridSearchCV, GroupKFold

from Springer_Segmentation.gerar_modelo import treinar_modelo_springer
from source.extrair_features import extrair_features

# Segmentação de Springer
O algoritmo proposto por Springer et al. (2016) é o estado da arte para a segmentação de sinais de fonocardiograma (PCG). Utilizando Modelos Ocultos de Markov com Duração Explícita (HSMM) e o algoritmo de Viterbi, o modelo analisa características do áudio (como Transformadas de Wavelet e Energia de Shannon) para delinear o ciclo cardíaco com precisão fisiológica.

Sua função neste projeto é receber o áudio bruto e classificar o sinal em quatro fases mecânicas fundamentais:

- S1 (Primeira Bulha): Início da sístole (fechamento das válvulas mitral e tricúspide).

- Sístole: Período de ejeção ventricular.

- S2 (Segunda Bulha): Início da diástole (fechamento das válvulas aórtica e pulmonar).

- Diástole: Período de relaxamento e enchimento.

A partir dessa marcação temporal confiável, torna-se possível extrair as métricas de duração e amplitude que alimentarão os classificadores de Machine Learning nas etapas seguintes.

## Gerando o Modelo de Segmentação de Springer

In [8]:
# Configure os caminhos do seu computador
CAMINHO_TCC = r"C:\Users\gusta\TCC\TCC"
PASTA_REPO = os.path.join(CAMINHO_TCC, "Springer_Segmentation")

# A pasta de treino do Physionet e onde o modelo vai nascer
PASTA_PHYSIONET = os.path.join(PASTA_REPO, "training_data")
MODELO_PKL = os.path.join(CAMINHO_TCC, "springer_segmentation_model.pkl")

modelo_final = treinar_modelo_springer(
    caminho_repo = PASTA_REPO,
    pasta_dados = PASTA_PHYSIONET,
    arquivo_saida = MODELO_PKL,
    max_audios = 1000
)

--- INICIANDO TREINAMENTO DO MODELO SPRINGER ---
1/4 - Lendo arquivos de treinamento (.wav e .tsv)...


 11%|█         | 1133/10431 [00:05<00:44, 208.42it/s]


KeyboardInterrupt: 

## Extraindo as features

In [ ]:
CAMINHO_TCC = r"C:\Users\gusta\TCC\TCC"
PASTA_REPO = os.path.join(CAMINHO_TCC, "Springer_Segmentation")
MODELO_PKL = r"springer_segmentation_model.pkl"
PASTA_AUDIOS = r"C:\Users\gusta\TCC\TCC\data\train"
ARQUIVO_SAIDA = os.path.join(CAMINHO_TCC, "features_extraidas.csv")

features_extraidas = extrair_features(
    caminho_repo = PASTA_REPO,
    caminho_modelo = MODELO_PKL,
    pasta_audios = PASTA_AUDIOS,
    arquivo_saida = ARQUIVO_SAIDA
)

features_extraidas.head()

### Fusão de Dados
Unificando as características temporais extraídas dos áudios com os rótulos clínicos e metadados dos pacientes. Como o arquivo original de treino possui os áudios dispostos em colunas (formato *wide*), utilizamos a técnica de *Melt* (unpivot) para transformar cada gravação em uma linha individual (formato *long*), permitindo o cruzamento exato com as *features* matemáticas.

In [ ]:
# --- 1. CARREGAMENTO E DERRETIMENTO (Melt) ---
caminho_features = 'features_extraidas.csv'
caminho_train = r'data\train.csv'
caminho_meta = r'data\additional_metadata.csv'

df_features = pd.read_csv(caminho_features)
df_train = pd.read_csv(caminho_train)
df_meta = pd.read_csv(caminho_meta)

colunas_audios = [f'recording_{i}' for i in range(1, 9)]
df_train_linhas = df_train.melt(
    id_vars=['patient_id', 'AS', 'AR', 'MR', 'MS', 'N'], 
    value_vars=colunas_audios, 
    value_name='nome_do_audio_sem_wav'
).dropna(subset=['nome_do_audio_sem_wav'])

df_train_linhas['arquivo_wav'] = df_train_linhas['nome_do_audio_sem_wav'].astype(str) + '.wav'
df_features = df_features.rename(columns={'patient_id': 'arquivo_wav'})

# --- 2. FUSÃO GERAL ---
df_rotulos = pd.merge(df_train_linhas, df_meta, on='patient_id', how='left')
dataset_final = pd.merge(df_features, df_rotulos, on='arquivo_wav', how='inner')
dataset_final = dataset_final.drop(columns=['variable', 'nome_do_audio_sem_wav'])

# --- 3. ENGENHARIA DE FEATURES (Foco e Postura) ---
def classificar_foco(nome_arquivo):
    nome = str(nome_arquivo).upper()
    if 'AOR' in nome: return 'Aortico'
    if 'PUL' in nome: return 'Pulmonar'
    if 'TRI' in nome: return 'Tricuspide'
    if 'MIR' in nome or 'MIT' in nome: return 'Mitral'
    return 'Indefinido'

def binario_postura(nome_arquivo):
    # 1 para Sentado (sit), 0 para Supino/Deitado (sup)
    return 1 if 'sit' in str(nome_arquivo).lower() else 0

print("A extrair Foco (4 colunas binárias) e Postura (1 coluna binária)...")
dataset_final['Foco'] = dataset_final['arquivo_wav'].apply(classificar_foco)
dataset_final['Postura_Sentado'] = dataset_final['arquivo_wav'].apply(binario_postura)

# One-Hot Encoding: Cria as 4 colunas independentes para o Random Forest (Aortico, Pulmonar, Tricuspide, Mitral)
dataset_final = pd.get_dummies(dataset_final, columns=['Foco'], prefix='Foco', dtype=int)

# Remover a coluna N (Redundância linear)
if 'N' in dataset_final.columns:
    dataset_final = dataset_final.drop(columns=['N'])

# --- 4. TRATAMENTO DE METADADOS NUMÉRICOS ---
dataset_final['Gender'] = dataset_final['Gender'].map({'M': 1, 'F': 0})
dataset_final['Lives'] = dataset_final['Lives'].map({'U': 1, 'R': 0})

def converter_idade(idade_str):
    if pd.isna(idade_str): return np.nan
    try:
        partes = str(idade_str).split('-')
        return (int(partes[0]) + int(partes[1])) / 2.0
    except:
        return np.nan 

dataset_final['Age'] = dataset_final['Age'].apply(converter_idade)

for col in ['Age', 'Gender', 'Smoker', 'Lives']:
    dataset_final[col] = dataset_final[col].fillna(dataset_final[col].mean())

# --- 5. REORGANIZAÇÃO ELEGANTE DAS COLUNAS ---
colunas_identificacao = ['patient_id', 'arquivo_wav']
colunas_alvo = ['AS', 'AR', 'MR', 'MS']
# Pegamos dinamicamente as colunas novas de Foco que foram geradas
colunas_foco = [c for c in dataset_final.columns if 'Foco_' in c]
colunas_metadados = ['Age', 'Gender', 'Smoker', 'Lives', 'Postura_Sentado'] + colunas_foco

colunas_atuais = dataset_final.columns.tolist()
colunas_features_temporais = [c for c in colunas_atuais if c not in colunas_identificacao + colunas_alvo + colunas_metadados]

colunas_organizadas = colunas_identificacao + colunas_alvo + colunas_metadados + colunas_features_temporais
dataset_final = dataset_final[colunas_organizadas]

dataset_final = dataset_final.dropna(how='any')

# --- 6. SALVAR E EXIBIR ---
caminho_salvar = 'dataset_final.csv'
dataset_final.to_csv(caminho_salvar, index=False)

print(f"Sucesso! Dataset super limpo e organizado salvo em: {caminho_salvar}\n")
dataset_final.head()

A extrair Foco (4 colunas binárias) e Postura (1 coluna binária)...
Sucesso! Dataset super limpo e organizado salvo em: dataset_final.csv



,patient_id,arquivo_wav,AS,AR,MR,MS,Age,Gender,Smoker,Lives,...,m_Ratio_SysRR,sd_Ratio_SysRR,m_Ratio_DiaRR,sd_Ratio_DiaRR,m_Ratio_SysDia,sd_Ratio_SysDia,m_Amp_SysS1,sd_Amp_SysS1,m_Amp_DiaS2,sd_Amp_DiaS2
0,patient_016,AR_016_sit_Aor.wav,0.0,1.0,0.0,0.0,34.5,1,1,1,...,0.091438,0.019738,0.660700,0.034412,0.140255,0.039258,0.412363,0.123084,0.884418,0.351987
1,patient_016,AR_016_sit_Mit.wav,0.0,1.0,0.0,0.0,34.5,1,1,1,...,0.107745,0.023489,0.641216,0.037205,0.169973,0.042578,0.391649,0.132505,0.898236,0.316727
2,patient_016,AR_016_sit_Pul.wav,0.0,1.0,0.0,0.0,34.5,1,1,1,...,0.222610,0.007361,0.526831,0.016333,0.423049,0.021189,0.201668,0.049161,0.787470,0.258047
3,patient_016,AR_016_sit_Tri.wav,0.0,1.0,0.0,0.0,34.5,1,1,1,...,0.217510,0.014600,0.539695,0.022940,0.404594,0.042584,0.279554,0.205512,0.716258,0.249561
4,patient_016,AR_016_sup_Aor.wav,0.0,1.0,0.0,0.0,34.5,1,1,1,...,0.042312,0.008571,0.704802,0.053228,0.061668,0.022319,0.357486,0.267772,0.834465,0.384031


### Seleção de Modelos (Benchmarking)
O objetivo principal deste projeto é diagnosticar simultaneamente a presença de quatro patologias valvulares (AS, AR, MR, MS) e a ausência delas (N). Como um paciente pode apresentar múltiplas condições concomitantes, o problema é modelado como Classificação Multilabel.

Para algoritmos que não suportam saídas múltiplas nativamente, adotamos a estratégia `MultiOutputClassifier`, que ajusta um classificador independente para cada classe. A avaliação foca no F1-Score Macro e, em especial, no PhysioNet Score Macro (a média aritmética entre Sensibilidade e Especificidade adaptada para múltiplas classes), uma métrica clínica rigorosa que recompensa a segurança de detectar a doença e penaliza falsos alarmes.

Nesta etapa, treinamos e avaliamos múltiplos algoritmos de Machine Learning na sua configuração padrão (baseline). O objetivo é identificar qual arquitetura consegue capturar melhor os padrões das características extraídas (Wavelets e durações de Liu et al.) para este cenário de diagnósticos múltiplos.

Para garantir uma comparação justa entre modelos baseados em árvores e modelos lineares/distância, os dados preditores (X) foram normalizados utilizando o `StandardScaler`.

Modelos Testados:
1. Random Forest (Floresta Aleatória)
2. Gradient Boosting (Árvores em Cascata)
3. Regressão Logística (Estatística Clássica)
4. Support Vector Machine - SVM (Fronteiras de Decisão)
5. K-Nearest Neighbors - KNN (Agrupamento por Similaridade)

In [11]:
# 1. Carregar Dados
caminho_dataset = 'dataset_final.csv'
df = pd.read_csv(caminho_dataset)
alvos = ['AS', 'AR', 'MR', 'MS'] # Lembrando que tiramos o 'N'

# 2. Isolar Pacientes (Anti-Leakage)
df_pacientes = df[['patient_id'] + alvos].drop_duplicates()
X_pat = df_pacientes[['patient_id']].to_numpy()
y_pat = df_pacientes[alvos].to_numpy()

X_pat_train, y_pat_train, X_pat_test, y_pat_test = iterative_train_test_split(X_pat, y_pat, test_size=0.2)
ids_treino = [item[0] for item in X_pat_train]
ids_teste = [item[0] for item in X_pat_test]

# -> AQUI NASCE O df_treino e o df_teste <-
df_treino = df[df['patient_id'].isin(ids_treino)].copy()
df_teste = df[df['patient_id'].isin(ids_teste)].copy()

# 3. Separar as features do gabarito
colunas_remover = ['patient_id', 'arquivo_wav'] + alvos
X_train = df_treino.drop(columns=colunas_remover)

# -> AQUI NASCE O y_train <-
y_train = df_treino[alvos]

X_test = df_teste.drop(columns=colunas_remover)
y_test = df_teste[alvos]

# 4. Padronização (Scaler)
scaler = StandardScaler()

# -> AQUI NASCE O X_train_scaled <-
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Dados preparados com sucesso! Variáveis carregadas na memória.")

✅ Dados preparados com sucesso! Variáveis carregadas na memória.


In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix

print("A iniciar o Benchmark Clínico Robusto (Métrica Ponderada)...")

# --- 1. A NOVA MÉTRICA CLÍNICA OFICIAL ---
def score_clinico_macro(y_true, y_pred, w_pos=5, w_neg=1):
    """
    Calcula o Score Clínico Ponderado (inspirado no PhysioNet 2022).
    Penaliza 5x mais a perda de um paciente doente (FN) do que um falso alarme (FP).
    """
    scores = []
    # Itera sobre as nossas 4 colunas alvo (AS, AR, MR, MS)
    for col in range(y_true.shape[1]):
        # O labels=[0,1] garante que o ravel não quebre se uma classe não aparecer num fold
        tn, fp, fn, tp = confusion_matrix(y_true.iloc[:, col], y_pred.iloc[:, col], labels=[0,1]).ravel()
        
        numerador = (w_pos * tp) + (w_neg * tn)
        denominador = (w_pos * (tp + fn)) + (w_neg * (tn + fp))
        
        score_doenca = numerador / denominador if denominador > 0 else 0
        scores.append(score_doenca)
        
    return np.mean(scores)

# --- 2. Preparação do Benchmark ---
modelos = {
    "Regressão Logística": MultiOutputClassifier(LogisticRegression(max_iter=2000, random_state=42)),
    "Gradient Boosting": MultiOutputClassifier(GradientBoostingClassifier(random_state=42)),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM (Vetores de Suporte)": MultiOutputClassifier(SVC(random_state=42)),
    "KNN": KNeighborsClassifier()
}

grupos_treino = df_treino['patient_id'].values
gkf = GroupKFold(n_splits=5)
resultados_cv = []

# --- 3. A Maratona de Testes (GroupKFold) ---
for nome_modelo, modelo in modelos.items():
    scores_folds = []
    
    # 5 Provas Simuladas
    for train_idx, val_idx in gkf.split(X_train_scaled, y_train, groups=grupos_treino):
        X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        pacientes_val = grupos_treino[val_idx]
        
        # Treina o modelo cego
        modelo.fit(X_fold_train, y_fold_train)
        pred_val = modelo.predict(X_fold_val)
        
        # Agregação Tardia por Paciente (Logical OR)
        df_val = pd.DataFrame(pred_val, columns=alvos)
        df_val['patient_id'] = pacientes_val
        y_val_paciente_pred = df_val.groupby('patient_id')[alvos].max()
        
        df_real = y_fold_val.copy()
        df_real['patient_id'] = pacientes_val
        y_val_paciente_real = df_real.groupby('patient_id')[alvos].max()
        
        # AVALIAÇÃO COM A NOVA MÉTRICA DE PESO 5
        score_fold = score_clinico_macro(y_val_paciente_real, y_val_paciente_pred)
        scores_folds.append(score_fold)
        
    # Estatísticas do modelo
    media_score = np.mean(scores_folds)
    desvio_score = np.std(scores_folds)
    
    resultados_cv.append({
        "Modelo": nome_modelo,
        "Score Clínico Ponderado (CV) %": round(media_score * 100, 2),
        "Estabilidade (±)": round(desvio_score * 100, 2)
    })

# --- 4. O Novo Pódio ---
df_resultados_cv = pd.DataFrame(resultados_cv)
df_resultados_cv = df_resultados_cv.sort_values(by="Score Clínico Ponderado (CV) %", ascending=False).reset_index(drop=True)

print("\n=== RANKING BASELINE DEFINITIVO (MÉTRICA CLÍNICA + CROSS-VALIDATION) ===")
display(df_resultados_cv.style.background_gradient(cmap='Oranges'))

A iniciar o Benchmark Clínico Robusto (Métrica Ponderada)...

=== RANKING BASELINE DEFINITIVO (MÉTRICA CLÍNICA + CROSS-VALIDATION) ===


,Modelo,Score Clínico Ponderado (CV) %,Estabilidade (±)
0,Gradient Boosting,81.200000,1.470000
1,Random Forest,80.580000,5.630000
2,KNN,80.230000,1.730000
3,SVM (Vetores de Suporte),76.760000,3.870000
4,Regressão Logística,75.360000,4.820000


## Tuning da Regressão Logística

In [14]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
import itertools
import numpy as np
import warnings

# Ignorar avisos de convergência
warnings.filterwarnings('ignore')

print("A iniciar o Tuning Customizado focado no Paciente (Regressão Logística)...")

# 1. Os mesmos grupos de pacientes do Treino
grupos_treino = df_treino['patient_id'].values

# 2. As combinações que queremos testar para a LR
parametros_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'class_weight': [None, 'balanced'],
    'solver': ['lbfgs', 'liblinear']
}

# Cria uma lista com todas as combinações possíveis
chaves, valores = zip(*parametros_lr.items())
combinacoes = [dict(zip(chaves, v)) for v in itertools.product(*valores)]

melhor_score_physionet_lr = 0
melhores_params_lr = None
gkf = GroupKFold(n_splits=5)

total_comb = len(combinacoes)
print(f"A testar {total_comb} combinações com Agregação Tardia Interna...")

# 3. O nosso Motor de Busca Manual
for i, params in enumerate(combinacoes):
    scores_folds = []
    
    for train_idx, val_idx in gkf.split(X_train_scaled, y_train, groups=grupos_treino):
        X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        pacientes_val = grupos_treino[val_idx]
        
        # Cria e treina o modelo com os parâmetros da vez
        # Aumentamos o max_iter para evitar erros de convergência
        modelo = MultiOutputClassifier(LogisticRegression(max_iter=2000, random_state=42, **params))
        modelo.fit(X_fold_train, y_fold_train)
        
        # Previsão nas linhas isoladas
        pred_val = modelo.predict(X_fold_val)
        
        # Agrega por paciente DENTRO do Tuning
        df_val = pd.DataFrame(pred_val, columns=alvos)
        df_val['patient_id'] = pacientes_val
        y_val_paciente_pred = df_val.groupby('patient_id')[alvos].max()
        
        df_real = y_fold_val.copy()
        df_real['patient_id'] = pacientes_val
        y_val_paciente_real = df_real.groupby('patient_id')[alvos].max()
        
        # Avalia usando o PhysioNet Score
        score_fold = score_physionet_macro(y_val_paciente_real, y_val_paciente_pred)
        scores_folds.append(score_fold)
        
    media_score = np.mean(scores_folds)
    
    if media_score > melhor_score_physionet_lr:
        melhor_score_physionet_lr = media_score
        melhores_params_lr = params

print("\n=== RESULTADOS DO TUNING CUSTOMIZADO (LR) ===")
print(f"Melhor PhysioNet Score (Validação Interna): {melhor_score_physionet_lr * 100:.2f}%")
print(f"Parâmetros Vencedores: {melhores_params_lr}")

# --- 4. TREINAR O CAMPEÃO DEFINITIVO E TESTAR NA PROVA FINAL ---
print("\nA treinar o Campeão Definitivo com todos os dados de treino...")
melhor_lr = MultiOutputClassifier(LogisticRegression(max_iter=2000, random_state=42, **melhores_params_lr))
melhor_lr.fit(X_train_scaled, y_train)

# Prova Final (Teste Cego)
predicoes_teste_lr = melhor_lr.predict(X_test_scaled)
df_previsoes_lr = pd.DataFrame(predicoes_teste_lr, columns=alvos)
df_previsoes_lr['patient_id'] = df_teste['patient_id'].values

y_test_paciente_pred_lr = df_previsoes_lr.groupby('patient_id')[alvos].max()

# O y_test_paciente_real já deve estar na memória, mas recalculamos por segurança
y_test_paciente_real = df_teste.groupby('patient_id')[alvos].max()

acc_exata_lr = accuracy_score(y_test_paciente_real, y_test_paciente_pred_lr)
f1_macro_lr = f1_score(y_test_paciente_real, y_test_paciente_pred_lr, average='macro', zero_division=0)
physionet_score_lr = score_physionet_macro(y_test_paciente_real, y_test_paciente_pred_lr)

print("=== PLACAR DEFINITIVO (REGRESSÃO LOGÍSTICA AFINADA) ===")
print(f"Acurácia Exata (Pacientes):  {acc_exata_lr * 100:.2f}%")
print(f"F1-Score Macro (Pacientes):  {f1_macro_lr * 100:.2f}%")
print(f"PhysioNet Score (Pacientes): {physionet_score_lr * 100:.2f}%")

A iniciar o Tuning Customizado focado no Paciente (Regressão Logística)...
A testar 20 combinações com Agregação Tardia Interna...

=== RESULTADOS DO TUNING CUSTOMIZADO (LR) ===
Melhor PhysioNet Score (Validação Interna): 68.17%
Parâmetros Vencedores: {'C': 10, 'class_weight': None, 'solver': 'lbfgs'}

A treinar o Campeão Definitivo com todos os dados de treino...
=== PLACAR DEFINITIVO (REGRESSÃO LOGÍSTICA AFINADA) ===
Acurácia Exata (Pacientes):  18.18%
F1-Score Macro (Pacientes):  62.79%
PhysioNet Score (Pacientes): 68.42%


## Tuning do Gradient Boosting

In [13]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score
import itertools
import numpy as np

print("A iniciar o Tuning Customizado focado no Paciente (Patient-Aware Tuning)...")

# 1. Os mesmos grupos de pacientes do Treino
grupos_treino = df_treino['patient_id'].values

# 2. As combinações que queremos testar
parametros_gb = {
    'n_estimators': [50, 100, 150, 200],
    'learning_rate': [0.05, 0.1, 0.2, 0.3],
    'max_depth': [3, 5]
}

# Cria uma lista com todas as combinações possíveis
chaves, valores = zip(*parametros_gb.items())
combinacoes = [dict(zip(chaves, v)) for v in itertools.product(*valores)]

# Variáveis para guardar o campeão
melhor_score_physionet = 0
melhores_params_gb = None
gkf = GroupKFold(n_splits=5)

total_comb = len(combinacoes)
print(f"A testar {total_comb} combinações com Agregação Tardia Interna...")

# 3. O nosso Motor de Busca Manual
for i, params in enumerate(combinacoes):
    scores_folds = []
    
    # Fazemos a Validação Cruzada (5 cortes)
    for train_idx, val_idx in gkf.split(X_train_scaled, y_train, groups=grupos_treino):
        X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        pacientes_val = grupos_treino[val_idx]
        
        # Cria e treina o modelo com os parâmetros da vez
        modelo = MultiOutputClassifier(GradientBoostingClassifier(random_state=42, **params))
        modelo.fit(X_fold_train, y_fold_train)
        
        # Previsão nas linhas isoladas do Fold de Validação
        pred_val = modelo.predict(X_fold_val)
        
        # --- A MÁGICA: Agrega por paciente DENTRO do Tuning ---
        df_val = pd.DataFrame(pred_val, columns=alvos)
        df_val['patient_id'] = pacientes_val
        y_val_paciente_pred = df_val.groupby('patient_id')[alvos].max()
        
        df_real = y_fold_val.copy()
        df_real['patient_id'] = pacientes_val
        y_val_paciente_real = df_real.groupby('patient_id')[alvos].max()
        
        # Avalia usando a nossa Métrica Clínica Oficial!
        score_fold = score_clinico_macro(y_val_paciente_real, y_val_paciente_pred)
        scores_folds.append(score_fold)
        
    # A média do PhysioNet Score nos 5 cortes
    media_score = np.mean(scores_folds)
    
    # Se bater o recorde, guardamos!
    if media_score > melhor_score_physionet:
        melhor_score_physionet = media_score
        melhores_params_gb = params

print("\n=== RESULTADOS DO TUNING CUSTOMIZADO ===")
print(f"Melhor PhysioNet Score (Validação Interna): {melhor_score_physionet * 100:.2f}%")
print(f"Parâmetros Vencedores: {melhores_params_gb}")

# --- 4. TREINAR O CAMPEÃO DEFINITIVO E TESTAR NA PROVA FINAL ---
print("\nA treinar o Campeão Definitivo com todos os dados de treino...")
melhor_gb = MultiOutputClassifier(GradientBoostingClassifier(random_state=42, **melhores_params_gb))
melhor_gb.fit(X_train_scaled, y_train)

# Prova Final
predicoes_teste_gb = melhor_gb.predict(X_test_scaled)
df_previsoes_gb = pd.DataFrame(predicoes_teste_gb, columns=alvos)
df_previsoes_gb['patient_id'] = df_teste['patient_id'].values

y_test_paciente_pred_gb = df_previsoes_gb.groupby('patient_id')[alvos].max()
y_test_paciente_real = df_teste.groupby('patient_id')[alvos].max()

acc_exata_gb = accuracy_score(y_test_paciente_real, y_test_paciente_pred_gb)
f1_macro_gb = f1_score(y_test_paciente_real, y_test_paciente_pred_gb, average='macro', zero_division=0)
physionet_score_gb = score_clinico_macro(y_test_paciente_real, y_test_paciente_pred_gb)

print("=== PLACAR DEFINITIVO (GRADIENT BOOSTING AFINADO) ===")
print(f"Acurácia Exata (Pacientes):  {acc_exata_gb * 100:.2f}%")
print(f"F1-Score Macro (Pacientes):  {f1_macro_gb * 100:.2f}%")
print(f"PhysioNet Score (Pacientes): {physionet_score_gb * 100:.2f}%")

A iniciar o Tuning Customizado focado no Paciente (Patient-Aware Tuning)...
A testar 32 combinações com Agregação Tardia Interna...

=== RESULTADOS DO TUNING CUSTOMIZADO ===
Melhor PhysioNet Score (Validação Interna): 76.64%
Parâmetros Vencedores: {'n_estimators': 150, 'learning_rate': 0.2, 'max_depth': 3}

A treinar o Campeão Definitivo com todos os dados de treino...
=== PLACAR DEFINITIVO (GRADIENT BOOSTING AFINADO) ===
Acurácia Exata (Pacientes):  27.27%
F1-Score Macro (Pacientes):  65.35%
PhysioNet Score (Pacientes): 82.93%


## Tuning Random Forest

In [14]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier
import itertools
import numpy as np
import pandas as pd

print("A iniciar o Tuning Customizado focado no Paciente (Random Forest)...")

# 1. Os mesmos grupos de pacientes do Treino
grupos_treino = df_treino['patient_id'].values

# 2. As combinações que queremos testar
parametros_rf = {
    'n_estimators': [100, 150, 200],
    'max_depth': [None, 5, 10],
    'class_weight': [None, 'balanced']
}

# Cria uma lista com todas as combinações possíveis
chaves, valores = zip(*parametros_rf.items())
combinacoes = [dict(zip(chaves, v)) for v in itertools.product(*valores)]

# Variáveis para guardar o campeão
melhor_score_physionet_rf = 0
melhores_params_rf = None
gkf = GroupKFold(n_splits=5)

total_comb = len(combinacoes)
print(f"A testar {total_comb} combinações com Agregação Tardia Interna...")

# 3. O nosso Motor de Busca Manual
for i, params in enumerate(combinacoes):
    scores_folds = []
    
    # Fazemos a Validação Cruzada (5 cortes)
    for train_idx, val_idx in gkf.split(X_train_scaled, y_train, groups=grupos_treino):
        X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        pacientes_val = grupos_treino[val_idx]
        
        # Cria e treina o modelo com os parâmetros da vez
        # O RF suporta multilabel nativamente, dispensando o MultiOutputClassifier
        modelo = RandomForestClassifier(random_state=42, **params)
        modelo.fit(X_fold_train, y_fold_train)
        
        # Previsão nas linhas isoladas do Fold de Validação
        pred_val = modelo.predict(X_fold_val)
        
        # --- A MÁGICA: Agrega por paciente DENTRO do Tuning ---
        df_val = pd.DataFrame(pred_val, columns=alvos)
        df_val['patient_id'] = pacientes_val
        y_val_paciente_pred = df_val.groupby('patient_id')[alvos].max()
        
        df_real = y_fold_val.copy()
        df_real['patient_id'] = pacientes_val
        y_val_paciente_real = df_real.groupby('patient_id')[alvos].max()
        
        # Avalia usando a nossa Métrica Clínica Oficial!
        score_fold = score_clinico_macro(y_val_paciente_real, y_val_paciente_pred)
        scores_folds.append(score_fold)
        
    # A média do PhysioNet Score nos 5 cortes
    media_score = np.mean(scores_folds)
    
    # Se bater o recorde, guardamos!
    if media_score > melhor_score_physionet_rf:
        melhor_score_physionet_rf = media_score
        melhores_params_rf = params

print("\n=== RESULTADOS DO TUNING CUSTOMIZADO (RANDOM FOREST) ===")
print(f"Melhor PhysioNet Score (Validação Interna): {melhor_score_physionet_rf * 100:.2f}%")
print(f"Parâmetros Vencedores: {melhores_params_rf}")

# --- 4. TREINAR O CAMPEÃO DEFINITIVO E TESTAR NA PROVA FINAL ---
print("\nA treinar o Campeão Definitivo com todos os dados de treino...")
melhor_rf = RandomForestClassifier(random_state=42, **melhores_params_rf)
melhor_rf.fit(X_train_scaled, y_train)

# Prova Final (O Cofre)
predicoes_teste_rf = melhor_rf.predict(X_test_scaled)
df_previsoes_rf = pd.DataFrame(predicoes_teste_rf, columns=alvos)
df_previsoes_rf['patient_id'] = df_teste['patient_id'].values

y_test_paciente_pred_rf = df_previsoes_rf.groupby('patient_id')[alvos].max()
y_test_paciente_real = df_teste.groupby('patient_id')[alvos].max()

acc_exata_rf = accuracy_score(y_test_paciente_real, y_test_paciente_pred_rf)
f1_macro_rf = f1_score(y_test_paciente_real, y_test_paciente_pred_rf, average='macro', zero_division=0)
physionet_score_rf = score_clinico_macro(y_test_paciente_real, y_test_paciente_pred_rf)

print("=== PLACAR DEFINITIVO (RANDOM FOREST AFINADO) ===")
print(f"Acurácia Exata (Pacientes):  {acc_exata_rf * 100:.2f}%")
print(f"F1-Score Macro (Pacientes):  {f1_macro_rf * 100:.2f}%")
print(f"PhysioNet Score (Pacientes): {physionet_score_rf * 100:.2f}%")

A iniciar o Tuning Customizado focado no Paciente (Random Forest)...
A testar 18 combinações com Agregação Tardia Interna...

=== RESULTADOS DO TUNING CUSTOMIZADO (RANDOM FOREST) ===
Melhor PhysioNet Score (Validação Interna): 78.08%
Parâmetros Vencedores: {'n_estimators': 150, 'max_depth': 10, 'class_weight': 'balanced'}

A treinar o Campeão Definitivo com todos os dados de treino...
=== PLACAR DEFINITIVO (RANDOM FOREST AFINADO) ===
Acurácia Exata (Pacientes):  22.73%
F1-Score Macro (Pacientes):  59.89%
PhysioNet Score (Pacientes): 74.20%


## Tuning do KNN

In [15]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.neighbors import KNeighborsClassifier
import itertools
import numpy as np
import pandas as pd

print("A iniciar o Tuning Customizado focado no Paciente (KNN)...")

# 1. Os mesmos grupos de pacientes do Treino
grupos_treino = df_treino['patient_id'].values

# 2. As combinações espaciais que queremos testar
parametros_knn = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}

# Cria uma lista com todas as combinações possíveis
chaves, valores = zip(*parametros_knn.items())
combinacoes = [dict(zip(chaves, v)) for v in itertools.product(*valores)]

# Variáveis para guardar o campeão
melhor_score_physionet_knn = 0
melhores_params_knn = None
gkf = GroupKFold(n_splits=5)

total_comb = len(combinacoes)
print(f"A testar {total_comb} combinações com Agregação Tardia Interna...")

# 3. O nosso Motor de Busca Manual
for i, params in enumerate(combinacoes):
    scores_folds = []
    
    # Fazemos a Validação Cruzada (5 cortes)
    for train_idx, val_idx in gkf.split(X_train_scaled, y_train, groups=grupos_treino):
        X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        pacientes_val = grupos_treino[val_idx]
        
        # Cria e treina o modelo com os parâmetros da vez
        modelo = KNeighborsClassifier(**params)
        modelo.fit(X_fold_train, y_fold_train)
        
        # Previsão nas linhas isoladas do Fold de Validação
        pred_val = modelo.predict(X_fold_val)
        
        # --- A MÁGICA: Agrega por paciente DENTRO do Tuning ---
        df_val = pd.DataFrame(pred_val, columns=alvos)
        df_val['patient_id'] = pacientes_val
        y_val_paciente_pred = df_val.groupby('patient_id')[alvos].max()
        
        df_real = y_fold_val.copy()
        df_real['patient_id'] = pacientes_val
        y_val_paciente_real = df_real.groupby('patient_id')[alvos].max()
        
        # Avalia usando a nossa Métrica Clínica Oficial!
        score_fold = score_clinico_macro(y_val_paciente_real, y_val_paciente_pred)
        scores_folds.append(score_fold)
        
    # A média do PhysioNet Score nos 5 cortes
    media_score = np.mean(scores_folds)
    
    # Se bater o recorde, guardamos!
    if media_score > melhor_score_physionet_knn:
        melhor_score_physionet_knn = media_score
        melhores_params_knn = params

print("\n=== RESULTADOS DO TUNING CUSTOMIZADO (KNN) ===")
print(f"Melhor PhysioNet Score (Validação Interna): {melhor_score_physionet_knn * 100:.2f}%")
print(f"Parâmetros Vencedores: {melhores_params_knn}")

# --- 4. TREINAR O CAMPEÃO DEFINITIVO E TESTAR NA PROVA FINAL ---
print("\nA treinar o Campeão Definitivo com todos os dados de treino...")
melhor_knn = KNeighborsClassifier(**melhores_params_knn)
melhor_knn.fit(X_train_scaled, y_train)

# Prova Final (O Cofre)
predicoes_teste_knn = melhor_knn.predict(X_test_scaled)
df_previsoes_knn = pd.DataFrame(predicoes_teste_knn, columns=alvos)
df_previsoes_knn['patient_id'] = df_teste['patient_id'].values

y_test_paciente_pred_knn = df_previsoes_knn.groupby('patient_id')[alvos].max()
y_test_paciente_real = df_teste.groupby('patient_id')[alvos].max()

acc_exata_knn = accuracy_score(y_test_paciente_real, y_test_paciente_pred_knn)
f1_macro_knn = f1_score(y_test_paciente_real, y_test_paciente_pred_knn, average='macro', zero_division=0)
physionet_score_knn = score_clinico_macro(y_test_paciente_real, y_test_paciente_pred_knn)

print("=== PLACAR DEFINITIVO (KNN AFINADO) ===")
print(f"Acurácia Exata (Pacientes):  {acc_exata_knn * 100:.2f}%")
print(f"F1-Score Macro (Pacientes):  {f1_macro_knn * 100:.2f}%")
print(f"PhysioNet Score (Pacientes): {physionet_score_knn * 100:.2f}%")

A iniciar o Tuning Customizado focado no Paciente (KNN)...
A testar 20 combinações com Agregação Tardia Interna...

=== RESULTADOS DO TUNING CUSTOMIZADO (KNN) ===
Melhor PhysioNet Score (Validação Interna): 79.91%
Parâmetros Vencedores: {'n_neighbors': 7, 'weights': 'distance', 'p': 2}

A treinar o Campeão Definitivo com todos os dados de treino...
=== PLACAR DEFINITIVO (KNN AFINADO) ===
Acurácia Exata (Pacientes):  18.18%
F1-Score Macro (Pacientes):  59.41%
PhysioNet Score (Pacientes): 75.19%


## Ensemble

In [16]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

print("A convocar a Junta Médica (100% Afinada pelo Tuning)...")

# 1. Instanciar os 3 modelos com os parâmetros exatos do GridSearch
modelo_gb_tunado = MultiOutputClassifier(GradientBoostingClassifier(
    n_estimators=150, 
    learning_rate=0.2, 
    max_depth=3, 
    random_state=42
))

modelo_rf_tunado = RandomForestClassifier(
    n_estimators=150, 
    max_depth=10, 
    class_weight='balanced', 
    random_state=42
)

modelo_knn_tunado = KNeighborsClassifier(
    n_neighbors=7, 
    weights='distance', 
    p=2
)

# 2. Treinar os especialistas com todos os dados
print("A treinar os especialistas com as hiper-configurações...")
modelo_gb_tunado.fit(X_train_scaled, y_train)
modelo_rf_tunado.fit(X_train_scaled, y_train)
modelo_knn_tunado.fit(X_train_scaled, y_train)

# 3. Previsões no Cofre (O Teste Cego Final)
print("A realizar os diagnósticos no grupo de Teste...")
pred_gb_audios = modelo_gb_tunado.predict(X_test_scaled)
pred_rf_audios = modelo_rf_tunado.predict(X_test_scaled)
pred_knn_audios = modelo_knn_tunado.predict(X_test_scaled)

# 4. Agregação por Paciente (A Regra de Triagem do MAX)
def agregar_paciente(predicoes, pacientes, colunas):
    df_temp = pd.DataFrame(predicoes, columns=colunas)
    df_temp['patient_id'] = pacientes
    return df_temp.groupby('patient_id')[colunas].max()

pacientes_teste = df_teste['patient_id'].values
y_pred_gb = agregar_paciente(pred_gb_audios, pacientes_teste, alvos)
y_pred_rf = agregar_paciente(pred_rf_audios, pacientes_teste, alvos)
y_pred_knn = agregar_paciente(pred_knn_audios, pacientes_teste, alvos)

# --- 5. A ASSEMBLEIA FINAL: VOTAÇÃO POR CONSENSO (>= 2 Votos) ---
print("Em votação: Se 2 médicos concordarem, o diagnóstico é confirmado.")
soma_votos = y_pred_gb + y_pred_rf + y_pred_knn
y_pred_ensemble = (soma_votos >= 2).astype(int)

# 6. Confrontar com o Gabarito
y_test_paciente_real = df_teste.groupby('patient_id')[alvos].max()

acc_ensemble = accuracy_score(y_test_paciente_real, y_pred_ensemble)
f1_ensemble = f1_score(y_test_paciente_real, y_pred_ensemble, average='macro', zero_division=0)
physionet_ensemble = score_clinico_macro(y_test_paciente_real, y_pred_ensemble)

print("\n=== PLACAR DO CONSELHO MÉDICO (100% TUNADO) ===")
print(f"Acurácia Exata (Pacientes):  {acc_ensemble * 100:.2f}%")
print(f"F1-Score Macro (Pacientes):  {f1_ensemble * 100:.2f}%")
print(f"PhysioNet Score (Pacientes): {physionet_ensemble * 100:.2f}%")

A convocar a Junta Médica (100% Afinada pelo Tuning)...
A treinar os especialistas com as hiper-configurações...
A realizar os diagnósticos no grupo de Teste...
Em votação: Se 2 médicos concordarem, o diagnóstico é confirmado.

=== PLACAR DO CONSELHO MÉDICO (100% TUNADO) ===
Acurácia Exata (Pacientes):  22.73%
F1-Score Macro (Pacientes):  63.19%
PhysioNet Score (Pacientes): 78.93%
